# Assignment #2: Language models
Author: Pierre Nugues

## Objectives

The objectives of this assignment are to:
* Write a program to find n-gram statistics
* Compute the probability of a sentence
* Know what a language model is
* Write a short report of 1 to 2 pages on the assignment

## Submission

Once you have written all the missing code and run all the cells, you will show it to an instructor that will pass you.

## Organization

* Each group will have to write Python programs to count unigrams, bigrams, and trigrams in a corpus of approximately one million words and to determine the probability of a sentence.
* You can test you regular expression using the regex101.com site
* Each student will have to write a short report of one to two pages and comment briefly the results. In your report, you must produce the tabulated results of your analysis as described below.

## Programming

### Imports

Some imports you may need. Add others as needed.

In [241]:
import math
import regex as re
import json

### Collecting and analyzing a corpus

Retrieve a corpus of novels by Selma Lagerl&ouml;f from this URL:
<a href="https://github.com/pnugues/ilppp/blob/master/programs/corpus/Selma.txt">
    <tt>https://github.com/pnugues/ilppp/blob/master/programs/corpus/Selma.txt</tt>
</a>. The text of these novels was extracted
from <a href="https://litteraturbanken.se/forfattare/LagerlofS/titlar">Lagerlöf arkivet</a> at
<a href="https://litteraturbanken.se/">Litteraturbanken</a>.

In [242]:
# You may have to adjust the path
corpus = open('Selma.txt', encoding='utf8').read()

Run the <a href="https://github.com/pnugues/ilppp/tree/master/programs/ch02/python">concordance
program </a> to print the lines containing a specific word, for instance <i>Nils</i>.

In [243]:
pattern = 'Nils Holgersson'
width = 25
# corpus = "Hej på dig Nils Holgersson, hur mår du? Jag hoppas att du har haft en bra dag. Nils Holgersson är en fantastisk karaktär i boken. Nils Holgersson har många äventyr framför sig. Nils Holgersson är känd för sin modighet och vänlighet."

In [244]:
# spaces match tabs and newlines
pattern = re.sub(' ', r'\\s+', pattern)
# Replaces newlines with spaces in the text
clean_corpus = re.sub(r'\s+', ' ', corpus)
concordance = ('(.{{0,{width}}}{pattern}.{{0,{width}}})'
               .format(pattern=pattern, width=width))
for match in re.finditer(concordance, clean_corpus):
    print(match.group(1))
# print the string with 0..width characters on either side

Selma Lagerlöf Nils Holgerssons underbara resa genom Sv
! Se på Tummetott! Se på Nils Holgersson Tummetott!» Genast vände
r,» sade han. »Jag heter Nils Holgersson och är son till en husma
lden. »Inte är det värt, Nils Holgersson, att du är ängslig eller
 i dem. På den tiden, då Nils Holgersson drog omkring med vildgäs
ulle allt visa honom vad Nils Holgersson från Västra Vemmenhög va
om ägde rum det året, då Nils Holgersson for omkring med vildgäss
m vad det kan kosta dem. Nils Holgersson hade inte haft förstånd 
de det inte mer sägas om Nils Holgersson, att han inte tyckte om 
 Rosenbom?» För där stod Nils Holgersson mitt uppe på Rosenboms n
 Med ens fingo de syn på Nils Holgersson, och då sköt den store v
vila. När vildgässen och Nils Holgersson äntligen hade letat sig 
 slags arbetare. Men vad Nils Holgersson inte såg, det var, att s
nde han fråga, och om då Nils Holgersson sade nej, började han ge
de lille Mats, och om nu Nils Holgersson också hade tegat, så had
åg så försmädlig ut,

Run a simple <a href="https://github.com/pnugues/ilppp/tree/master/programs/ch05/python">tokenization
program</a> on your corpus.

In [245]:
def tokenize(text):
    words = re.findall(r'\p{L}+', text)
    return words

In [246]:
words = tokenize(corpus)
words[:10]
# len(words)

['Selma',
 'Lagerlöf',
 'Nils',
 'Holgerssons',
 'underbara',
 'resa',
 'genom',
 'Sverige',
 'Första',
 'bandet']

Count the number of unique words in the original corpus and when setting all the words in lowercase

Original text

In [247]:
# Write your code here
len(set(words))

44256

Lowercased text

In [248]:
# Write your code here
len(set(map(lambda x : x.lower(), words)))

41032

### Segmenting a corpus

You will write a program to tokenize your text, insert `<s>` and `</s>` tags to delimit sentences, and set all the words in lowercase letters. In the end, you will only keep the words.

#### Normalizing 

Write a regular expression that matches all the characters that are neither a letter nor a punctuation sign. The punctuations signs will be the followings: `.;:?!`. In your regex, use the same order. For the definition of a letter, use a Unicode regex. You will call the regex string `nonletter`

In [249]:
# Write your code
nonletter = r"[^.;:?! \p{L}]"

Write a `clean()` function that replaces all the characters that are neither a letter nor a punctuation sign with a space. The punctuations signs will be the followings: `.;:?!`.   For the sentence:

_En gång hade de på Mårbacka en barnpiga, som hette Back-Kajsa._

the result will be:

`En gång hade de på Mårbacka en barnpiga som hette Back Kajsa.`

In [250]:
# Write your code here
def clean(text):
    """
    Tokenize words
    """
    text_nonletters_replaced = " ".join(re.split(nonletter, text))
    text_dubbelspeces_removed = re.sub(r'[\p{Zs}]+'," ", text_nonletters_replaced)
    return text_dubbelspeces_removed

In [251]:
test_para = 'En gång hade de på Mårbacka en barnpiga, som hette Back-Kajsa. \
Hon var nog sina tre alnar lång, hon hade ett stort, grovt ansikte med stränga, mörka drag, \
hennes händer voro hårda och fulla av sprickor, som barnens hår fastnade i, \
när hon kammade dem, och till humöret var hon dyster och sorgbunden.'

In [252]:
test_para_2 = "Hej, mitt namn är Axel. Jag gillar godis och Kalle-Anka! Varför inte spela Minecraft? Man kan spela tennis!"

In [253]:
test = re.split(nonletter,test_para)
test = " ".join(test)
test = re.sub(r'[ ]+', " ", test)
print(test)

En gång hade de på Mårbacka en barnpiga som hette Back Kajsa. Hon var nog sina tre alnar lång hon hade ett stort grovt ansikte med stränga mörka drag hennes händer voro hårda och fulla av sprickor som barnens hår fastnade i när hon kammade dem och till humöret var hon dyster och sorgbunden.


In [254]:
test_para = clean(test_para)
test_para

'En gång hade de på Mårbacka en barnpiga som hette Back Kajsa. Hon var nog sina tre alnar lång hon hade ett stort grovt ansikte med stränga mörka drag hennes händer voro hårda och fulla av sprickor som barnens hår fastnade i när hon kammade dem och till humöret var hon dyster och sorgbunden.'

In [255]:
test_para_2 = clean(test_para_2)
test_para_2

'Hej mitt namn är Axel. Jag gillar godis och Kalle Anka! Varför inte spela Minecraft? Man kan spela tennis!'

In [256]:
print(corpus[:200])

Selma Lagerlöf


Nils Holgerssons underbara resa genom Sverige. Första bandet





Bokutgåva


Albert Bonniers förlag, Stockholm 1907.




DEN KRISTLIGA DAGVISAN.





Den signade dag, som vi nu här s


In [257]:
print(clean(corpus[:200]))

Selma Lagerlöf Nils Holgerssons underbara resa genom Sverige. Första bandet Bokutgåva Albert Bonniers förlag Stockholm . DEN KRISTLIGA DAGVISAN. Den signade dag som vi nu här s


#### Segmenter

In this section, you will write a sentence segmenter that will delimit each sentence with `</s>` and `<s>` symbols. For example the sentence:

_En gång hade de på Mårbacka en barnpiga, som hette Back-Kajsa._

will be bracketed as:

`<s> En gång hade de på Mårbacka en barnpiga som hette Back-Kajsa </s>`

As algorithm, you will use a simple heuristics to detect the sentence boundaries: A sentence starts with a capital letter and ends with a period-equivalent punctuation sign. You will write a regex to match these boundaries with a regular expression and you will insert `</s>\n<s>` symbols with a substitution function.

##### Detecting sentence boundaries

Write a regular expression that matches a punctuation, a sequence of spaces, and an uppercase letter. Call this regex string `sentence_boundaries`. In the regex, you will remember the value of the uppercase letter using a backreference. Use the Unicode regexes for the letters and the spaces. You may use `\s` though.

In [258]:
# Write your code here
# pun
sentence_boundaries = r'[.;:?!][\p{Zs}]+(\p{Lu})'

##### Replacement markup

Write a string to replace the matched boundaries with the sentence boundary markup. Remember that a sentence ends with `</s>` and starts with `<s>` and that there is one sentence per line. Hint: The markup is `</s>\n<s>`. Remember also that the first letter of your sentence is in a regex backreference. Call the regex string `sentence_markup`.

In [259]:
# Write your code here
sentence_markup = " </s>\n<s> "

##### Applying the substitution

Use your regexes to segment your text. Use the string `sentence_boundaries`, `sentence_markup`, and `test_para` as input and `text` as output.

In [260]:
# Write your code here
text = re.sub(sentence_boundaries, sentence_markup + r'\1', test_para)

In [262]:
print(text)

En gång hade de på Mårbacka en barnpiga som hette Back Kajsa </s>
<s> Hon var nog sina tre alnar lång hon hade ett stort grovt ansikte med stränga mörka drag hennes händer voro hårda och fulla av sprickor som barnens hår fastnade i när hon kammade dem och till humöret var hon dyster och sorgbunden.


Insert markup codes in the beginning and end of the text

In [263]:
# Write your code here
text = "<s> " + text + " </s>"

In [264]:
print(text)

<s> En gång hade de på Mårbacka en barnpiga som hette Back Kajsa </s>
<s> Hon var nog sina tre alnar lång hon hade ett stort grovt ansikte med stränga mörka drag hennes händer voro hårda och fulla av sprickor som barnens hår fastnade i när hon kammade dem och till humöret var hon dyster och sorgbunden. </s>


Replace the space duplicates with one space and remove the punctuation signs. For the spaces, use the Unicode regex. You may use `\s` though.

In [265]:
# Write your code here
text = re.sub(r'[\p{Zs}.;:?!]+', " ", text)

In [266]:
print(text)

<s> En gång hade de på Mårbacka en barnpiga som hette Back Kajsa </s>
<s> Hon var nog sina tre alnar lång hon hade ett stort grovt ansikte med stränga mörka drag hennes händer voro hårda och fulla av sprickor som barnens hår fastnade i när hon kammade dem och till humöret var hon dyster och sorgbunden </s>


Write a `segment_sentences(text)` function to gather the code in the Segmenter section and set the text in lowercase

In [267]:
# Write your code here
def segment_sentences(text):
    text_without_beg_end = re.sub(sentence_boundaries, sentence_markup + r'\1', text)
    text_with_beg_end = "<s> " + text_without_beg_end + "</s>"
    text_no_dubblespaces = re.sub(r'[\p{Zs}.;:?!]+', " ", text_with_beg_end)
    return text_no_dubblespaces.lower()

In [268]:
print(segment_sentences(test_para))

<s> en gång hade de på mårbacka en barnpiga som hette back kajsa </s>
<s> hon var nog sina tre alnar lång hon hade ett stort grovt ansikte med stränga mörka drag hennes händer voro hårda och fulla av sprickor som barnens hår fastnade i när hon kammade dem och till humöret var hon dyster och sorgbunden </s>


Estimate qualitatively the accuracy of your program.

#### Tokenizing the corpus

Clean and segment the corpus using the functions you have written

In [269]:
# Write your code here
corpus = segment_sentences(clean(corpus))

The result should be a normalized text without punctuation signs where all the sentences are delimited with `<s>` and `</s>` tags. The five last lines of the text should look like this. You may have some small differences.

In [271]:
print(corpus[-557:])

<s> hon hade fått större kärlek av sina föräldrar än någon annan han visste och sådan kärlek måste vändas i välsignelse </s>
<s> då prästen sade detta kom alla människor att se bort mot klara gulla och de förundrade sig över vad de såg </s>
<s> prästens ord tycktes redan ha gått i uppfyllelse </s>
<s> där stod klara fina gulleborg ifrån skrolycka hon som var uppkallad efter själva solen vid sina föräldrars grav och lyste som en förklarad </s>
<s> hon var likaså vacker som den söndagen då hon gick till kyrkan i den röda klänningen om inte vackrare </s>


You will now create a list of words from your string. You will consider that a space or a carriage return is an item separator

In [272]:
# Write your code here
words = re.split(r'[ \n]',corpus)

The five last lines of the corpus should like this:

In [274]:
print(words[-101:])

['<s>', 'hon', 'hade', 'fått', 'större', 'kärlek', 'av', 'sina', 'föräldrar', 'än', 'någon', 'annan', 'han', 'visste', 'och', 'sådan', 'kärlek', 'måste', 'vändas', 'i', 'välsignelse', '</s>', '<s>', 'då', 'prästen', 'sade', 'detta', 'kom', 'alla', 'människor', 'att', 'se', 'bort', 'mot', 'klara', 'gulla', 'och', 'de', 'förundrade', 'sig', 'över', 'vad', 'de', 'såg', '</s>', '<s>', 'prästens', 'ord', 'tycktes', 'redan', 'ha', 'gått', 'i', 'uppfyllelse', '</s>', '<s>', 'där', 'stod', 'klara', 'fina', 'gulleborg', 'ifrån', 'skrolycka', 'hon', 'som', 'var', 'uppkallad', 'efter', 'själva', 'solen', 'vid', 'sina', 'föräldrars', 'grav', 'och', 'lyste', 'som', 'en', 'förklarad', '</s>', '<s>', 'hon', 'var', 'likaså', 'vacker', 'som', 'den', 'söndagen', 'då', 'hon', 'gick', 'till', 'kyrkan', 'i', 'den', 'röda', 'klänningen', 'om', 'inte', 'vackrare', '</s>']


### Counting unigrams and bigrams

Read and try programs to compute the frequency of unigrams and bigrams of the training set in the noteboo: [https://github.com/pnugues/pnlp/blob/main/notebooks/10_01_ngrams.ipynb](https://github.com/pnugues/pnlp/blob/main/notebooks/10_01_ngrams.ipynb).

#### Unigrams

In [275]:
def unigrams(words):
    frequency = {}
    for i in range(len(words)):
        if words[i] in frequency:
            frequency[words[i]] += 1
        else:
            frequency[words[i]] = 1
    return frequency

We compute the frequencies.

In [276]:
frequency = unigrams(words)
list(frequency.items())[:20]

[('<s>', 59047),
 ('selma', 52),
 ('lagerlöf', 270),
 ('nils', 87),
 ('holgerssons', 6),
 ('underbara', 23),
 ('resa', 317),
 ('genom', 688),
 ('sverige', 56),
 ('</s>', 59047),
 ('första', 525),
 ('bandet', 6),
 ('bokutgåva', 11),
 ('albert', 15),
 ('bonniers', 11),
 ('förlag', 11),
 ('stockholm', 77),
 ('den', 11624),
 ('kristliga', 2),
 ('dagvisan', 2)]

#### Bigrams

In [277]:
def bigrams(words):
    bigrams = []
    for i in range(len(words) - 1):
        bigrams.append((words[i], words[i + 1]))
    frequency_bigrams = {}
    for i in range(len(words) - 1):
        if bigrams[i] in frequency_bigrams:
            frequency_bigrams[bigrams[i]] += 1
        else:
            frequency_bigrams[bigrams[i]] = 1
    return frequency_bigrams

In [279]:
frequency_bigrams = bigrams(words)
list(frequency_bigrams.items())[:20]

[(('<s>', 'selma'), 8),
 (('selma', 'lagerlöf'), 11),
 (('lagerlöf', 'nils'), 1),
 (('nils', 'holgerssons'), 6),
 (('holgerssons', 'underbara'), 4),
 (('underbara', 'resa'), 4),
 (('resa', 'genom'), 6),
 (('genom', 'sverige'), 5),
 (('sverige', '</s>'), 17),
 (('</s>', '<s>'), 59046),
 (('<s>', 'första'), 11),
 (('första', 'bandet'), 1),
 (('bandet', 'bokutgåva'), 2),
 (('bokutgåva', 'albert'), 11),
 (('albert', 'bonniers'), 11),
 (('bonniers', 'förlag'), 11),
 (('förlag', 'stockholm'), 10),
 (('stockholm', '</s>'), 24),
 (('<s>', 'den'), 1375),
 (('den', 'kristliga'), 2)]

In the report, tell what is the possible number of bigrams and their real number? Explain why such a difference. What would be the possible number of 4-grams.

Propose a solution to cope with bigrams unseen in the corpus.

### Computing the likelihood of a sentence
You will now compute the likelihood of a sentence using a bigram and a trigram models. 

In both models, you will ignore the start of sentence symbol, `<s>`, as this factor is common to both models: $P(<s>)$. This will save you one multiplication.

#### Unigrams

Write a program to compute a sentence's probability using unigrams. Your function will return the perplexity.

Your function should print and tabulate the results as in the examples below with the sentence _Det var en gång en katt som hette Nils_.

Your figures might be slightly different because of differences in the sentence segmentation.

```
=====================================================
wi 	 C(wi) 	 #words 	 P(wi)
=====================================================
det 	 21108 	 1041631 	 0.0202643738521607
var 	 12090 	 1041631 	 0.01160679741674355
en 	 13514 	 1041631 	 0.01297388422579589
gång 	 1332 	 1041631 	 0.001278763784871994
en 	 13514 	 1041631 	 0.01297388422579589
katt 	 16 	 1041631 	 1.5360525944408337e-05
som 	 16288 	 1041631 	 0.015637015411407686
hette 	 97 	 1041631 	 9.312318853797554e-05
nils 	 87 	 1041631 	 8.352285982272032e-05
</s> 	 59047 	 1041631 	 0.056687060964967444
=====================================================
Prob. unigrams:	 5.361459667285409e-27
Geometric mean prob.: 0.0023600885848765307
Entropy rate:	 8.726943273141258
Perplexity:	 423.71290908655254
```

In [373]:
# Write your code
def unigram_lm(frequency, sent_words):
    """
    Computing the sentence probs with a unigram model.
    """
    print('Unigram model')
    print('======================================================')
    print(f'{"wi":<10}{"C(wi)":<10}{"#words":<10}{"P(wi)":<10}')
    print('======================================================')
    total_count = sum(frequency.values())
    inv_word_count = 1/len(sent_words)
    prob_unigrams = 1
    prob_log = 0
    for word in sent_words :
        count = frequency.get(word, 0)
        prob = count/total_count
        if prob != 0 :
            prob_unigrams *= prob
            prob_log += math.log2(prob)
        print(f"{word:<10}{count:<10}{total_count:<10}{prob:<10}")
    entropy = -1*prob_log*inv_word_count
    perplexity = math.pow(2,entropy)
    print('======================================================')
    print(f"{"Prob. unigrams:":<30}{prob_unigrams}")
    print(f"{"Geometric mean prob.:":<30}{math.pow(prob_unigrams,inv_word_count)}")
    print(f"{"Entropy rate:":<30}{entropy}")
    print(f"{"Perplexity:":<30}{perplexity}")
    return perplexity

    

In [374]:
sentence = 'det var en gång en katt som hette nils </s>'
sent_words = sentence.split()
sent_words

['det', 'var', 'en', 'gång', 'en', 'katt', 'som', 'hette', 'nils', '</s>']

In [375]:
perplexity_unigrams = unigram_lm(frequency, sent_words)

Unigram model
wi        C(wi)     #words    P(wi)     
det       21108     1041579   0.020265385534846612
var       12090     1041579   0.011607376876837956
en        13514     1041579   0.012974531936607785
gång      1332      1041579   0.0012788276261330154
en        13514     1041579   0.012974531936607785
katt      16        1041579   1.5361292806402586e-05
som       16288     1041579   0.015637796076917832
hette     97        1041579   9.312783763881568e-05
nils      87        1041579   8.352702963481407e-05
</s>      59047     1041579   0.056689891021228345
Prob. unigrams:               5.364136934636431e-27
Geometric mean prob.:         0.0023602064104148853
Entropy rate:                 8.726871249541004
Perplexity:                   423.6917566138702


In [35]:
perplexity_unigrams = unigram_lm(frequency, sent_words)

Unigram model
wi 	 C(wi) 	 #words 	 P(wi)
det 	 21108 	 1041631 	 0.0202643738521607
var 	 12090 	 1041631 	 0.01160679741674355
en 	 13514 	 1041631 	 0.01297388422579589
gång 	 1332 	 1041631 	 0.001278763784871994
en 	 13514 	 1041631 	 0.01297388422579589
katt 	 16 	 1041631 	 1.5360525944408337e-05
som 	 16288 	 1041631 	 0.015637015411407686
hette 	 97 	 1041631 	 9.312318853797554e-05
nils 	 87 	 1041631 	 8.352285982272032e-05
</s> 	 59047 	 1041631 	 0.056687060964967444
Prob. unigrams:	 5.361459667285409e-27 
Geometric mean prob.: 0.0023600885848765307 
Entropy rate:	 8.726943273141258 
Perplexity:	 423.71290908655254 



In [331]:
perplexity_unigrams = int(perplexity_unigrams)
perplexity_unigrams

423

#### Bigrams

Write a program to compute the sentence probability using bigrams. Your function will tabulate and print the results as below. It will return the perplexity.

```
=====================================================
wi 	 wi+1 	 Ci,i+1 	 C(i) 	 P(wi+1|wi)
=====================================================
<s>	 det 	 5672 	 59047 	 0.09605907158704083
det 	 var 	 3839 	 21108 	 0.1818741709304529
var 	 en 	 712 	 12090 	 0.058891645988420185
en 	 gång 	 706 	 13514 	 0.052242119283705785
gång 	 en 	 20 	 1332 	 0.015015015015015015
en 	 katt 	 6 	 13514 	 0.0004439840165754033
katt 	 som 	 2 	 16 	 0.125
som 	 hette 	 45 	 16288 	 0.002762770137524558
hette 	 nils 	 0 	 97 	 0.0 	 *backoff: 	 8.352285982272032e-05
nils 	 </s> 	 2 	 87 	 0.022988505747126436
=====================================================
Prob. bigrams:	 2.376007803503683e-19
Geometric mean prob.: 0.013727289294133601
Entropy rate:	 6.186809422848149
Perplexity:	 72.84759420254609
```

In [386]:
# Write your code
def bigram_lm(frequency, frequency_bigrams, sent_words):
    sent_words = ["<s>"] + sent_words
    print('Bigram model')
    print('==================================================================')
    print(f'{"wi":<10}{"wi+1":<10}{"C(wi+1,wi)":<15}{"C(wi)":<10}{"P(wi+1|wi)":<10}')
    print('==================================================================')
    total_count_unigrams = sum(frequency.values())
    total_count_bigrams = sum(frequency_bigrams.values())
    inv_word_count = 1/(len(sent_words)-1)
    prob_ = 1
    prob_log = 0
    for i in range(len(sent_words)-1) :
        backoff = False
        count_unigram = frequency.get(sent_words[i], 0)
        # prob_unigram = count_unigram/total_count_unigrams
        count_bigram = frequency_bigrams.get((sent_words[i],sent_words[i+1]),0)
        prob_bigram = count_bigram/count_unigram
        if prob_bigram == 0 :
            backoff = True
            prob_backoff = frequency.get(sent_words[i+1],0)/total_count_unigrams
            prob_ *= prob_backoff
            prob_log += math.log2(prob_backoff)
        else :
            prob_ *= prob_bigram
            prob_log += math.log2(prob_bigram)
        print(f"{sent_words[i]:<10}{sent_words[i+1]:<10}{count_bigram:<15}{count_unigram:<10}{prob_bigram:<10}{"Backoff: " if backoff else "":<}{prob_backoff if backoff else "":<}")
    entropy = -1*prob_log*inv_word_count
    perplexity = math.pow(2,entropy)
    print('==================================================================')    
    print(f"{"Prob. unigrams:":<30}{prob_}")
    print(f"{"Geometric mean prob.:":<30}{math.pow(prob_,inv_word_count)}")
    print(f"{"Entropy rate:":<30}{entropy}")
    print(f"{"Perplexity:":<30}{perplexity}")
    return perplexity
    

In [387]:
perplexity_bigrams = bigram_lm(frequency, frequency_bigrams, sent_words)

Bigram model
wi        wi+1      C(wi+1,wi)     C(wi)     P(wi+1|wi)
<s>       det       5672           59047     0.09605907158704083
det       var       3839           21108     0.1818741709304529
var       en        712            12090     0.058891645988420185
en        gång      706            13514     0.052242119283705785
gång      en        20             1332      0.015015015015015015
en        katt      6              13514     0.0004439840165754033
katt      som       2              16        0.125     
som       hette     45             16288     0.002762770137524558
hette     nils      0              97        0.0       Backoff: 8.352702963481407e-05
nils      </s>      2              87        0.022988505747126436
Prob. unigrams:               2.376126423796318e-19
Geometric mean prob.:         0.013727357824989852
Entropy rate:                 6.186802220488124
Perplexity:                   72.84723052673391


In [ ]:
perplexity_bigrams = bigram_lm(frequency, frequency_bigrams, sent_words)

Bigram model
wi        wi+1      C(wi+1,wi)     C(wi)     P(wi+1|wi)
<s>       det       5672           59047     0.09605907158704083
det       var       3839           21108     0.1818741709304529
var       en        712            12090     0.058891645988420185
en        gång      706            13514     0.052242119283705785
gång      en        20             1332      0.015015015015015015
en        katt      6              13514     0.0004439840165754033
katt      som       2              16        0.125     
som       hette     45             16288     0.002762770137524558
hette     nils      0              97        0.0       Backoff: 8.352702963481407e-05
nils      </s>      2              87        0.022988505747126436
Prob. unigrams:               2.376126423796318e-19
Geometric mean prob.:         0.013727357824989852
Entropy rate:                 6.186802220488124
Perplexity:                   72.84723052673391


In [371]:
perplexity_bigrams = int(perplexity_bigrams)
perplexity_bigrams

72

In addition to this sentence, _Det var en gång en katt som hette Nils_, write two other sentences that will form your test set and run your programs on them. You will insert them in your report.

In [389]:
sentence_1 = 'ibland är det bra att ha alla skedarna på rätt plats </s>'
sentence_2 = 'jag tycker det är rättvist att alla ska får någonstans att bo </s>'

sent_words_1 = sentence_1.split()
sent_words_2 = sentence_2.split()

perplexity_unigrams = unigram_lm(frequency, sent_words_1)
print()
perplexity_bigrams = bigram_lm(frequency, frequency_bigrams, sent_words_1)

Unigram model
wi        C(wi)     #words    P(wi)     
ibland    207       1041579   0.00019873672568283346
är        6290      1041579   0.006038908234517017
det       21108     1041579   0.020265385534846612
bra       466       1041579   0.0004473976529864753
att       28020     1041579   0.02690146402721253
ha        2131      1041579   0.0020459321856527444
alla      2355      1041579   0.002260990284942381
skedarna  1         1041579   9.600808004001616e-07
på        14250     1041579   0.013681151405702304
rätt      716       1041579   0.0006874178530865158
plats     244       1041579   0.00023425971529763946
</s>      59047     1041579   0.056689891021228345
Prob. unigrams:               1.623698738070265e-34
Geometric mean prob.:         0.001528300614604056
Entropy rate:                 9.353855937268744
Perplexity:                   654.3215323243681

Bigram model
wi        wi+1      C(wi+1,wi)     C(wi)     P(wi+1|wi)
<s>       ibland    35             59047     0.0005927481

### Online prediction of words

You will now carry out an online prediction of words. You will consider two cases:
1. Prediction of the current word a user is typing;
2. Prediction of the next word.

Ideally, you would write a loop that reads the words and apply the models while typing. As the Jupyter labs are not designed for interactive input and output, we will simplify the experimental settings with constant strings at a given time of the input.  

We will assume the user is typing the phrase: _Det var en gång_. 

#### Trigrams

To have a more accurate prediction, you will use a trigram counting function. Program this function following the model of your bigram counting function.

In [397]:
# Write your code
def trigrams(words):
    trigrams = []
    for i in range(len(words)-2) :
        trigrams.append((words[i],words[i+1],words[i+2]))
    frequency_trigrams = {}
    for i in range(len(words) - 2):
        if trigrams[i] in frequency_trigrams:
            frequency_trigrams[trigrams[i]] += 1
        else:
            frequency_trigrams[trigrams[i]] = 1
    return frequency_trigrams

In [398]:
frequency_trigrams = trigrams(words)
frequency_trigrams[('det', 'var', 'en')]

330

In [399]:
def export_ngrams_jsonl(dictionary, file_name):
    with open(file_name, "w", encoding="utf-8") as f:
        for k, v in dictionary.items():
            line = {"ngram": [k] if isinstance(k, str) else list(k),
                    "count": v}
            f.write(json.dumps(line, ensure_ascii=False) + "\n")
    print(f"'{file_name}' saved.")

In [400]:
export_ngrams_jsonl(frequency, "unigrams.jsonl")
export_ngrams_jsonl(frequency_bigrams, "bigrams.jsonl")
export_ngrams_jsonl(frequency_trigrams, "trigrams.jsonl")

'unigrams.jsonl' saved.
'bigrams.jsonl' saved.
'trigrams.jsonl' saved.


In [401]:
def load_ngrams_jsonl(file_name):
    dictionary = {}
    with open(file_name, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            k = tuple(data["ngram"]) if len(data["ngram"]) > 1 else data["ngram"][0]
            dictionary[k] = data["count"]
    print(
        f"'{file_name}' loaded with {len(dictionary)} elements.")
    return dictionary

In [402]:
frequency = load_ngrams_jsonl("unigrams.jsonl")
frequency_bigrams = load_ngrams_jsonl("bigrams.jsonl")
frequency_trigrams = load_ngrams_jsonl("trigrams.jsonl")

'unigrams.jsonl' loaded with 41034 elements.
'bigrams.jsonl' loaded with 320129 elements.
'trigrams.jsonl' loaded with 671273 elements.


In [403]:
frequency_bigrams

{('<s>', 'selma'): 8,
 ('selma', 'lagerlöf'): 11,
 ('lagerlöf', 'nils'): 1,
 ('nils', 'holgerssons'): 6,
 ('holgerssons', 'underbara'): 4,
 ('underbara', 'resa'): 4,
 ('resa', 'genom'): 6,
 ('genom', 'sverige'): 5,
 ('sverige', '</s>'): 17,
 ('</s>', '<s>'): 59046,
 ('<s>', 'första'): 11,
 ('första', 'bandet'): 1,
 ('bandet', 'bokutgåva'): 2,
 ('bokutgåva', 'albert'): 11,
 ('albert', 'bonniers'): 11,
 ('bonniers', 'förlag'): 11,
 ('förlag', 'stockholm'): 10,
 ('stockholm', '</s>'): 24,
 ('<s>', 'den'): 1375,
 ('den', 'kristliga'): 2,
 ('kristliga', 'dagvisan'): 2,
 ('dagvisan', '</s>'): 1,
 ('den', 'signade'): 2,
 ('signade', 'dag'): 2,
 ('dag', 'som'): 43,
 ('som', 'vi'): 113,
 ('vi', 'nu'): 24,
 ('nu', 'här'): 9,
 ('här', 'se'): 2,
 ('se', 'av'): 2,
 ('av', 'himmelen'): 1,
 ('himmelen', 'till'): 3,
 ('till', 'oss'): 57,
 ('oss', 'nedkomma'): 2,
 ('nedkomma', 'han'): 1,
 ('han', 'blive'): 1,
 ('blive', 'oss'): 1,
 ('oss', 'säll'): 1,
 ('säll', 'han'): 1,
 ('han', 'låte'): 1,
 ('låte',

#### Prediction

The user starts typing _Det var en gång_. After the 2nd character, your program tries to help the user with suggested words.

In [404]:
starting_text = 'De'.lower()
starting_text

'de'

Write a program to rank the five first candidates at this point. Assign these predictions in a list that you will call `current_word_predictions_1`. Note that you are starting a sentence and you can then use the bigram frequencies. Write a sorting key that will enable you to have a deterministic ranking of the words or bigrams with identical frequencies: When two words have the same frequency, you will sort them by alphabetic order. You can do this with a tuple.

In [417]:
cand_nbr = 5
frequency

{'<s>': 59047,
 'selma': 52,
 'lagerlöf': 270,
 'nils': 87,
 'holgerssons': 6,
 'underbara': 23,
 'resa': 317,
 'genom': 688,
 'sverige': 56,
 '</s>': 59047,
 'första': 525,
 'bandet': 6,
 'bokutgåva': 11,
 'albert': 15,
 'bonniers': 11,
 'förlag': 11,
 'stockholm': 77,
 'den': 11624,
 'kristliga': 2,
 'dagvisan': 2,
 'signade': 3,
 'dag': 942,
 'som': 16288,
 'vi': 2105,
 'nu': 4084,
 'här': 2107,
 'se': 1989,
 'av': 5435,
 'himmelen': 80,
 'till': 9139,
 'oss': 974,
 'nedkomma': 2,
 'han': 21589,
 'blive': 1,
 'säll': 7,
 'låte': 2,
 'sig': 9250,
 'te': 220,
 'alla': 2355,
 'glädje': 294,
 'och': 36356,
 'fromma': 29,
 'ja': 938,
 'herren': 23,
 'högste': 7,
 'i': 16508,
 'för': 9443,
 'synder': 6,
 'sorger': 15,
 'bevare': 18,
 'men': 8144,
 'såsom': 271,
 'en': 13514,
 'fågel': 49,
 'mot': 1526,
 'himmelens': 28,
 'höjd': 23,
 'lyfter': 29,
 'på': 14250,
 'lediga': 10,
 'vingar': 70,
 'lovar': 17,
 'sin': 2513,
 'gud': 532,
 'är': 6290,
 'glad': 436,
 'förnöjd': 8,
 'när': 2772,
 '

In [412]:
"he" > "ih"

False

In [428]:
# Write your code here
# candidates = [(key,value) for key, value in frequency_bigrams.items() if key[0] == starting_text]
candidates = [(key[1],value) for key, value in frequency_bigrams.items() if key[0] == "<s>" and key[1].startswith(starting_text)]

# print(candidates)
# def sorted_key(k) :
#     (x,y) = k
#     x_value = x[1]
#     y_value = y[1]
#     if x_value == y_value :
#         return x[0][1] > y[0][1]
#     return x_value > y_value
sorted_candidates = sorted(iter(candidates),key=lambda x : (1/x[1],x[0]))
print(sorted_candidates)
current_word_predictions_1 = [sorted_candidates[i][0] for i in range(5)]

[('det', 5672), ('de', 2071), ('den', 1375), ('detta', 298), ('denna', 80), ('dessa', 62), ('denne', 30), ('deras', 29), ('dessutom', 9), ('detsamma', 5), ('dels', 4), ('dem', 4), ('dess', 2), ('delfiner', 1), ('delta', 1), ('demonerna', 1), ('dervischen', 1), ('desamma', 1), ('desto', 1)]


In [429]:
current_word_predictions_1

['det', 'de', 'den', 'detta', 'denna']

Let us now suppose that the user has typed: _Det var en_. After detecting a space, your program starts predicting a next possible word.

In [430]:
current_text = "Det var en ".lower()
current_text

'det var en '

Tokenize this text and return a list of tokens. Call it `tokens`.

In [431]:
# Write your code here
tokens = current_text.split()

In [434]:
tokens
frequency_trigrams

{('<s>', 'selma', 'lagerlöf'): 5,
 ('selma', 'lagerlöf', 'nils'): 1,
 ('lagerlöf', 'nils', 'holgerssons'): 1,
 ('nils', 'holgerssons', 'underbara'): 4,
 ('holgerssons', 'underbara', 'resa'): 4,
 ('underbara', 'resa', 'genom'): 4,
 ('resa', 'genom', 'sverige'): 4,
 ('genom', 'sverige', '</s>'): 3,
 ('sverige', '</s>', '<s>'): 17,
 ('</s>', '<s>', 'första'): 11,
 ('<s>', 'första', 'bandet'): 1,
 ('första', 'bandet', 'bokutgåva'): 1,
 ('bandet', 'bokutgåva', 'albert'): 2,
 ('bokutgåva', 'albert', 'bonniers'): 11,
 ('albert', 'bonniers', 'förlag'): 11,
 ('bonniers', 'förlag', 'stockholm'): 10,
 ('förlag', 'stockholm', '</s>'): 9,
 ('stockholm', '</s>', '<s>'): 24,
 ('</s>', '<s>', 'den'): 1375,
 ('<s>', 'den', 'kristliga'): 2,
 ('den', 'kristliga', 'dagvisan'): 2,
 ('kristliga', 'dagvisan', '</s>'): 1,
 ('dagvisan', '</s>', '<s>'): 1,
 ('<s>', 'den', 'signade'): 2,
 ('den', 'signade', 'dag'): 2,
 ('signade', 'dag', 'som'): 2,
 ('dag', 'som', 'vi'): 3,
 ('som', 'vi', 'nu'): 5,
 ('vi', 'nu',

Write a program to propose the five next possible words ranked by frequency using a trigram model. Assign these predictions to a variable that you will call `next_word_predictions`. Write a sorting key that will enable you have a deterministic ranking of the words or bigrams with identical frequencies: When two words have the same frequency, you will sort them by alphabetic order. You can do this with a tuple.

In [439]:
tokens[-1]
for key, value in frequency_trigrams.items() :
    print(key[0])
    break

<s>


In [442]:
# Write your code here
candidates = [(key[2],value) for key, value in frequency_trigrams.items() if key[0] == tokens[-2] and key[1] == tokens[-1]]
print(candidates)
sorted_candidates = sorted(iter(candidates),key=lambda x : (1/x[1],x[0]))
print(sorted_candidates)
next_word_predictions = [sorted_candidates[i][0] for i in range(5)]

[('gång', 10), ('lycka', 5), ('stor', 34), ('ung', 13), ('människa', 4), ('storrövare', 1), ('som', 6), ('kringbyggd', 1), ('besynnerlig', 5), ('präktig', 5), ('milstolpe', 1), ('hederssak', 1), ('skara', 1), ('mästare', 3), ('riktig', 13), ('märkvärdig', 3), ('fullt', 1), ('drummel', 1), ('landstrykare', 1), ('djärv', 1), ('vådlig', 1), ('kväll', 2), ('hög', 5), ('stad', 1), ('kyrka', 2), ('grov', 1), ('fågel', 1), ('låg', 1), ('gås', 1), ('gammal', 19), ('god', 17), ('orimligt', 2), ('liten', 26), ('vacker', 7), ('lika', 1), ('kråkhanne', 1), ('sådan', 17), ('av', 16), ('flock', 1), ('tomte', 1), ('förfärlig', 4), ('ungfågel', 1), ('docka', 1), ('prakt', 1), ('skön', 4), ('älgtjur', 1), ('tagg', 1), ('hop', 1), ('blandning', 1), ('hund', 1), ('väta', 1), ('vindil', 1), ('trettifem', 1), ('mycket', 7), ('vind', 1), ('snökant', 1), ('gruvarbetare', 1), ('gruva', 1), ('lång', 8), ('smula', 8), ('förfärligt', 1), ('så', 15), ('örn', 2), ('fjällsjö', 1), ('vårbäck', 1), ('glad', 1), ('eft

In [443]:
next_word_predictions

['stor', 'liten', 'gammal', 'god', 'sådan']

Finally, let us suppose that the user has typed _Det var en g_, rank the five possible candidates. Assign these predictions in a list that you will call `current_word_predictions_2`

In [445]:
current_text = "Det var en g".lower()

In [447]:
# Write your code here
tokens = current_text.split()
candidates = [(key[2],value) for key, value in frequency_trigrams.items() if key[0] == tokens[-3] and key[1] == tokens[-2] and key[2].startswith(tokens[-1])]
print(candidates)
sorted_candidates = sorted(iter(candidates),key=lambda x : (1/x[1],x[0]))
print(sorted_candidates)
current_word_predictions_2 = [sorted_candidates[i][0] for i in range(5)]

[('gång', 10), ('grov', 1), ('gås', 1), ('gammal', 19), ('god', 17), ('gruvarbetare', 1), ('gruva', 1), ('glad', 1), ('gammaldags', 1), ('graf', 1), ('gengångare', 1), ('getabock', 1), ('gåva', 1), ('glänsande', 1), ('grann', 2), ('gränsbo', 1), ('ganska', 3), ('glädje', 2), ('gagnlös', 1), ('godmodig', 1), ('gård', 1), ('gosse', 1), ('gast', 1), ('gudsförnekare', 1), ('glittrande', 1), ('gråvädersdag', 1)]
[('gammal', 19), ('god', 17), ('gång', 10), ('ganska', 3), ('glädje', 2), ('grann', 2), ('gagnlös', 1), ('gammaldags', 1), ('gast', 1), ('gengångare', 1), ('getabock', 1), ('glad', 1), ('glittrande', 1), ('glänsande', 1), ('godmodig', 1), ('gosse', 1), ('graf', 1), ('grov', 1), ('gruva', 1), ('gruvarbetare', 1), ('gränsbo', 1), ('gråvädersdag', 1), ('gudsförnekare', 1), ('gård', 1), ('gås', 1), ('gåva', 1)]


In [448]:
current_word_predictions_2

['gammal', 'god', 'gång', 'ganska', 'glädje']

## Turning in your assignment

Now your are done with the program. To complete this assignment, you will:
1. Upload the ngram JSONL files in a Hugging Face dataset and create a space to predict the next word using trigrams. An elementary program will suffice. You can look at your teacher examples here: https://huggingface.co/datasets/pnugues/selma_ngrams and here https://huggingface.co/spaces/pnugues/selma_lm.
2. Write a short individual report on your program. I suggest that you use this structure for your report:
      1. Objectives and dataset
      2. Method and program structure, where you should outline your program and possibly describe difficult parts. In this section, you will include the __regular expression you used to segment the text__.
      3. Results. You will include the __unigram and bigram tables__ for _Det var en gång en katt som hette Nils_ and __two other sentences__. You will also include the results of your __next word prediction__.
      4. Conclusion
      5. Answer to possible questions
2. Execute the Jupyter notebook by Peter Norvig here: <a href="http://nbviewer.jupyter.org/url/norvig.com/ipython/How%20to%20Do%20Things%20with%20Words.ipynb">https://nbviewer.jupyter.org/url/norvig.com/ipython/How to Do Things with Words.ipynb</a>. Just run all the cells and be sure that you understand the code. You will find the data here: <a href="http://norvig.com/ngrams/">http://norvig.com/ngrams/</a>.
3. In your report, after the description of your program and conclusion, you will describe one experiment with Norvig's notebook and a __long string of words your will create yourself or copy from a text you like__. You will remove all the punctuation and white spaces from this string. You will set this string in lowercase letters. You will just add a cell at the end of Sect. 7 in Norvig's notebook, where you will use your string and run the notebook cell with the <tt>segment()</tt> and <tt>segment2()</tt> functions. __You will comment the segmentation results you obtained__ with the unigram and bigram models. You will give the URL of your Hugging Face space.

Submit your report as well as your **notebook** (for archiving purposes) to Canvas: https://canvas.education.lu.se/. To write your report, use Latex. This will probably help you structure your text. You can use the Overleaf online editor (www.overleaf.com). You will then upload a PDF file in Canvas.

The submission deadline is September 25, 2026. You will have only three submission attempts. The deadline for the second and third ones are one week after you are noticed of your result.